### Models

Purpose : Isolate modelling in one file 
This file only uses srcB which changes how we set the horizons. no need to open backend files.

In [1]:

import pandas as pd
import numpy as np
from math import sqrt, pi, exp
from arch import arch_model
import yfinance as yf 
from scipy.optimize import minimize
import warnings
from arch.__future__ import reindexing
from arch.utility.exceptions import ConvergenceWarning

from tqdm import tqdm

from xgboost import XGBRegressor
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="y is poorly scaled")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

import sys, os
sys.modules.pop("srcA", None)
sys.modules.pop("srcB", None)
os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.path.insert(0, "/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration")
sys.path.insert(0, "/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
from srcB.models_MFIV import estimate_naive10_mfiv, estimate_garch_mfiv

In [2]:
sys.modules.pop("srcB", None)

<module 'srcB' (namespace) from ['/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration/srcB']>

In [3]:
# saved in data prep 
import pandas as pd
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/x_by_year_fut/x_by_year.pkl", "rb") as f:
    X_by_year = pd.read_pickle(f)

In [6]:
X_by_year[2020].columns

Index(['rt', 'bsIV_2', 'MFIV_2', 'bsIV_3', 'MFIV_3', 'bsIV_4', 'MFIV_4',
       'bsIV_5', 'MFIV_5', 'bsIV_6',
       ...
       'MFIV_573', 'bsIV_574', 'MFIV_574', 'bsIV_575', 'MFIV_575', 'bsIV_576',
       'MFIV_576', 'mfiv_avg', 'mfiv_avg', 'rv'],
      dtype='str', length=1154)

In [5]:
YEARS = [2019, 2020, 2021]
for y in YEARS:
    X_by_year[y] = X_by_year[y].copy()
    X_by_year[y]["rv"] = X_by_year[y]["rt"] ** 2


### quarter split

In [7]:
# Just prepping for the short parameter tuning 

Xq_by_year = {}
YEARS = [2019, 2020, 2021]
for y in YEARS:
    Xq_by_year[y] = {}
    Xq_by_year[y]["q1"] = X_by_year[y].loc[f"{y}-01-01" : f"{y}-03-31 23:59:59"]
    Xq_by_year[y]["q2"] = X_by_year[y].loc[f"{y}-04-01" : f"{y}-06-30 23:59:59"]
    Xq_by_year[y]["q3"] = X_by_year[y].loc[f"{y}-07-01" : f"{y}-09-30 23:59:59"]
    Xq_by_year[y]["q4"] = X_by_year[y].loc[f"{y}-10-01" : f"{y}-12-31 23:59:59"]


### Outputs By Year

In [8]:
import pandas as pd
YEARS = [2019, 2020, 2021]
QUARTER = ["q1", "q2", "q3", "q4"]
outputs_by_year = {
    y : {
        q : pd.DataFrame(
            {
                "rv" : Xq_by_year[y][q]["rv"],
                "rt" : Xq_by_year[y][q]["rt"],
                "date_time" : Xq_by_year[y][q].index,
            }
        ) for q in QUARTER
    } for y in YEARS
}

In [9]:
YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]
# HOURS = [7, 46, 191]
# HOURS = [191, 382, 573, 764]
HOURS = list(range(2, 577))
for y in YEARS:
    for q in QUARTERS:
        cols = [f"MFIV_{H}" for H in HOURS]
        outputs_by_year[y][q] = pd.concat([outputs_by_year[y][q], Xq_by_year[y][q][cols]], axis = 1)


###  garch

In [10]:
from srcB.models import estimate_garch 

In [11]:
# # Prepare the dictionaries. Designed to be overwritten
# YEARS = [2022, 2023, 2024, 2025]
# QUARTERS = ["q1", "q2", "q3", "q4"]

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

idx_by_year = {y : {} for y in YEARS}
f_rollB_by_year = {y: {} for y in YEARS}
rolling_forecasts =  {y: {} for y in YEARS}
rolling_residuals = {y: {} for y in YEARS}
y_pred_by_year = {y: {} for y in YEARS}
y_true_by_year = {y: {} for y in YEARS}


In [12]:
# # change estimation to year. SKIP THIS 
# for y in YEARS:
#     for q in QUARTERS:

#         Xq_df = Xq_by_year[y][q]

#         f_rollB, y_true, idx = estimate_garch(Xq_df, lookback = 10)

#         dt = Xq_df.index
#         x_plotB = dt[idx]

#         assert len(x_plotB) == len(f_rollB) == len(y_true), (y, q)

#         idx_by_year[y][q] = idx
#         f_rollB_by_year[y][q] = f_rollB
#         rolling_forecasts[y][q] = f_rollB
#         rolling_residuals[y][q] = y_true - f_rollB
#         y_pred_by_year[y][q] = f_rollB
#         y_true_by_year[y][q] = y_true
        
#         outputs_by_year[y][q]["garch"] = pd.Series(f_rollB, Xq_by_year[y][q].index[idx])

In [13]:
# YEARLY ESTIMATION FOR GARCH
YEARS = [2019, 2020, 2021]
for y in YEARS:
    X_df = pd.concat([Xq_by_year[y][q] for q in QUARTERS]).sort_index()
    f_rollB, y_true, idx = estimate_garch(X_df, lookback = 10)
    dt = X_df.index
    x_plotB = dt[idx]
    
    assert len(x_plotB) == len(f_rollB) == len(y_true), y
    
    full_forecast = pd.Series(f_rollB, index = X_df.index[idx])
    full_true = pd.Series(y_true, index = X_df.index[idx])
    
    for q in QUARTERS:
        q_index = Xq_by_year[y][q].index
        f_q = full_forecast.reindex(q_index).dropna()
        y_q = full_true.reindex(f_q.index)
        
        idx_q = Xq_by_year[y][q].index.get_indexer(f_q.index)
        idx_by_year[y][q] = idx_q
        f_rollB_by_year[y][q] = f_q.to_numpy()
        rolling_forecasts[y][q] = f_q.to_numpy()
        rolling_residuals[y][q] = (y_q - f_q).to_numpy()
        y_pred_by_year[y][q] = f_q.to_numpy()
        y_true_by_year[y][q] = y_q.to_numpy()

        outputs_by_year[y][q]["garch"] = f_q

In [14]:
outputs_by_year[2021]["q2"].garch

date_time
2021-04-01 00:00:00+00:00    0.000093
2021-04-01 01:00:00+00:00    0.000058
2021-04-01 02:00:00+00:00    0.000093
2021-04-01 03:00:00+00:00    0.000065
2021-04-01 04:00:00+00:00    0.000034
                               ...   
2021-06-30 19:00:00+00:00    0.000077
2021-06-30 20:00:00+00:00    0.000080
2021-06-30 21:00:00+00:00    0.000075
2021-06-30 22:00:00+00:00    0.000071
2021-06-30 23:00:00+00:00    0.000063
Freq: h, Name: garch, Length: 2184, dtype: float64

In [15]:
os.makedirs("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year", exist_ok = True)

In [16]:
import pickle
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_01.pkl", "wb") as f:
    pickle.dump(outputs_by_year, f)


load

In [ ]:
import pickle 
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_01.pkl", "rb") as f:
    outputs_by_year  = pickle.load(f)

In [17]:

outputs_by_year[2021]["q1"]

,rv,rt,date_time,MFIV_2,MFIV_3,MFIV_4,MFIV_5,MFIV_6,MFIV_7,MFIV_8,...,MFIV_568,MFIV_569,MFIV_570,MFIV_571,MFIV_572,MFIV_573,MFIV_574,MFIV_575,MFIV_576,garch
date_time,,,,,,,,,,,,,,,,,,,,,
2021-01-01 00:00:00+00:00,1.161073e-05,-0.003407,2021-01-01 00:00:00+00:00,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,...,1.066339,1.066339,1.066339,1.066339,1.066339,1.066339,1.066339,1.066339,1.066339,NaN
2021-01-01 01:00:00+00:00,1.454392e-06,0.001206,2021-01-01 01:00:00+00:00,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,...,1.066339,1.066339,1.066339,1.066339,1.066339,1.066339,1.066339,1.066339,1.066339,NaN
2021-01-01 02:00:00+00:00,2.782889e-04,0.016682,2021-01-01 02:00:00+00:00,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,...,1.017498,1.017498,1.017498,1.017498,1.017498,1.017498,1.017498,1.017498,1.017498,NaN
2021-01-01 03:00:00+00:00,6.549349e-05,-0.008093,2021-01-01 03:00:00+00:00,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,...,0.970894,0.970894,0.970894,0.970894,0.970894,0.970894,0.970894,0.970894,0.970894,NaN
2021-01-01 04:00:00+00:00,1.173290e-05,0.003425,2021-01-01 04:00:00+00:00,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,2.261892,...,0.929171,0.929171,0.929171,0.929171,0.929171,0.929171,0.929171,0.929171,0.929171,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-03-31 19:00:00+00:00,4.932710e-06,0.002221,2021-03-31 19:00:00+00:00,0.750310,0.750310,0.750310,0.750310,0.750310,0.750310,0.750310,...,0.688046,0.688046,0.688046,0.688046,0.688046,0.688046,0.688046,0.688046,0.688046,0.000034
2021-03-31 20:00:00+00:00,2.861346e-04,-0.016916,2021-03-31 20:00:00+00:00,0.759894,0.759894,0.759894,0.759894,0.759894,0.759894,0.759894,...,0.643084,0.643084,0.643084,0.643084,0.643084,0.643084,0.643084,0.643084,0.643084,0.000030
2021-03-31 21:00:00+00:00,1.207885e-04,0.010990,2021-03-31 21:00:00+00:00,0.769600,0.769600,0.769600,0.769600,0.769600,0.769600,0.769600,...,0.601061,0.601061,0.601061,0.601061,0.601061,0.601061,0.601061,0.601061,0.601061,0.000101


### garch MFIV_191


In [17]:
# # Prepare the dictionaries. Designed to be overwritten
# YEARS = [2022, 2023, 2024, 2025]
# QUARTERS = ["q1", "q2", "q3", "q4"]

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

idx_by_year = {y : {} for y in YEARS}
f_rollB_by_year = {y: {} for y in YEARS}
rolling_forecasts =  {y: {} for y in YEARS}
rolling_residuals = {y: {} for y in YEARS}
y_pred_by_year = {y: {} for y in YEARS}
y_true_by_year = {y: {} for y in YEARS}


In [18]:
from srcB.models_MFIV import estimate_garch_mfiv

In [19]:
# YEARS = [2019, 2020, 2021]
# QUARTERS = ["q1", "q2", "q3", "q4"]

# for y in YEARS:
#     for q in QUARTERS:
#         Xq_df = Xq_by_year[y][q]

#         f_rollB, y_true, idx = estimate_garch_mfiv(
#             Xq_df,
#             exog_col="MFIV_191",
#             lookback=10,
#             correction_lookback=10
#         )

#         idx_by_year[y][q] = idx
#         f_rollB_by_year[y][q] = f_rollB
#         rolling_forecasts[y][q] = f_rollB
#         rolling_residuals[y][q] = y_true - f_rollB
#         y_pred_by_year[y][q] = f_rollB
#         y_true_by_year[y][q] = y_true

#         outputs_by_year[y][q]["garch_mfiv"] = pd.Series(
#             f_rollB,
#             index=Xq_df.index[idx]
#         )

In [20]:
# YEARLY ESTIMATION
YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

for y in YEARS:
    X_df = pd.concat([Xq_by_year[y][q] for q in QUARTERS]).sort_index()

    f_rollB, y_true, idx = estimate_garch_mfiv(
        X_df,
        exog_col="MFIV_191",
        lookback=10,
        correction_lookback=10
    )

    full_forecast = pd.Series(f_rollB, index=X_df.index[idx])
    full_true = pd.Series(y_true, index=X_df.index[idx])

    for q in QUARTERS:
        q_index = Xq_by_year[y][q].index
        f_q = full_forecast.reindex(q_index).dropna()
        y_q = full_true.reindex(f_q.index)

        idx_q = Xq_by_year[y][q].index.get_indexer(f_q.index)
        idx_by_year[y][q] = idx_q
        f_rollB_by_year[y][q] = f_q.to_numpy()
        rolling_forecasts[y][q] = f_q.to_numpy()
        rolling_residuals[y][q] = (y_q - f_q).to_numpy()
        y_pred_by_year[y][q] = f_q.to_numpy()
        y_true_by_year[y][q] = y_q.to_numpy()

        outputs_by_year[y][q]["garch_mfiv"] = f_q

In [21]:
import pickle
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year_02.pkl", "wb") as f:
    pickle.dump(outputs_by_year, f)


load

In [12]:
import pickle 
with open("/Users/shah/CODE_BOOK_4/CodeCleanup_II/script_data/IS/outputs_by_year/outputs_by_year_02.pkl", "rb") as f:
    outputs_by_year = pickle.load(f)

### garch MFIV avg

In [22]:
# # Prepare the dictionaries. Designed to be overwritten
# YEARS = [2022, 2023, 2024, 2025]
# QUARTERS = ["q1", "q2", "q3", "q4"]

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

idx_by_year = {y : {} for y in YEARS}
f_rollB_by_year = {y: {} for y in YEARS}
rolling_forecasts =  {y: {} for y in YEARS}
rolling_residuals = {y: {} for y in YEARS}
y_pred_by_year = {y: {} for y in YEARS}
y_true_by_year = {y: {} for y in YEARS}


In [23]:
from srcB.models_MFIV import estimate_garch_mfiv

In [24]:
# YEARS = [2019, 2020, 2021]
# QUARTERS = ["q1", "q2", "q3", "q4"]

# for y in YEARS:
#     for q in QUARTERS:
#         Xq_df = Xq_by_year[y][q]

#         f_rollB, y_true, idx = estimate_garch_mfiv(
#             Xq_df,
#             exog_col="mfiv_avg",
#             lookback=10,
#             correction_lookback=10
#         )

#         idx_by_year[y][q] = idx
#         f_rollB_by_year[y][q] = f_rollB
#         rolling_forecasts[y][q] = f_rollB
#         rolling_residuals[y][q] = y_true - f_rollB
#         y_pred_by_year[y][q] = f_rollB
#         y_true_by_year[y][q] = y_true

#         outputs_by_year[y][q]["garch_mfiv_avg"] = pd.Series(
#             f_rollB,
#             index=Xq_df.index[idx]
#         )

In [ ]:
print((X_df.columns == "mfiv_avg").sum())
print(X_df.loc[:, X_df.columns == "mfiv_avg"].shape)

2
(8760, 2)


In [30]:
YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

for y in YEARS:
    for q in QUARTERS:
        Xq_by_year[y][q] = Xq_by_year[y][q].loc[:, ~Xq_by_year[y][q].columns.duplicated(keep="first")]

In [31]:
YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

for y in YEARS:
    X_df = pd.concat([Xq_by_year[y][q] for q in QUARTERS]).sort_index()

    f_rollB, y_true, idx = estimate_garch_mfiv( X_df, exog_col="mfiv_avg", lookback=10, correction_lookback=10)

    full_forecast = pd.Series(f_rollB, index=X_df.index[idx])
    full_true = pd.Series(y_true, index=X_df.index[idx])

    for q in QUARTERS:
        q_index = Xq_by_year[y][q].index
        f_q = full_forecast.reindex(q_index).dropna()
        y_q = full_true.reindex(f_q.index)

        idx_q = Xq_by_year[y][q].index.get_indexer(f_q.index)
        idx_by_year[y][q] = idx_q
        f_rollB_by_year[y][q] = f_q.to_numpy()
        rolling_forecasts[y][q] = f_q.to_numpy()
        rolling_residuals[y][q] = (y_q - f_q).to_numpy()
        y_pred_by_year[y][q] = f_q.to_numpy()
        y_true_by_year[y][q] = y_q.to_numpy()

        outputs_by_year[y][q]["garch_mfiv_avg"] = f_q

In [32]:
# NOW CONTAINS THREE MODELS, garch, garch_mfiv(191), garch_mfiv_avg
import pickle
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_03.pkl", "wb") as f:
    pickle.dump(outputs_by_year, f)


In [41]:
import pickle 
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_03.pkl", "rb") as f:
    outputs_by_year = pickle.load(f)

In [30]:
outputs_by_year[2020]["q1"][20:200]

,rv,rt,date_time,MFIV_2,MFIV_3,MFIV_4,MFIV_5,MFIV_6,MFIV_7,MFIV_8,...,MFIV_570,MFIV_571,MFIV_572,MFIV_573,MFIV_574,MFIV_575,MFIV_576,garch,garch_mfiv,garch_mfiv_avg
date_time,,,,,,,,,,,,,,,,,,,,,
2020-01-01 20:00:00+00:00,1.732862e-06,-0.001316,2020-01-01 20:00:00+00:00,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,...,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.000003,NaN,NaN
2020-01-01 21:00:00+00:00,9.411596e-07,0.000970,2020-01-01 21:00:00+00:00,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,...,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.000003,NaN,NaN
2020-01-01 22:00:00+00:00,3.888139e-07,-0.000624,2020-01-01 22:00:00+00:00,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,...,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.000003,NaN,NaN
2020-01-01 23:00:00+00:00,3.014151e-05,-0.005490,2020-01-01 23:00:00+00:00,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,...,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.000003,NaN,NaN
2020-01-02 00:00:00+00:00,0.000000e+00,0.000000,2020-01-02 00:00:00+00:00,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,0.265246,...,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.355703,0.000004,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-01-09 03:00:00+00:00,1.132088e-06,-0.001064,2020-01-09 03:00:00+00:00,0.406456,0.406456,0.406456,0.406456,0.406456,0.406456,0.406456,...,0.465610,0.465610,0.465610,0.465610,0.465610,0.465610,0.465610,0.000092,7.563530e-05,8.076282e-05
2020-01-09 04:00:00+00:00,1.023231e-05,-0.003199,2020-01-09 04:00:00+00:00,0.404696,0.404696,0.404696,0.404696,0.404696,0.404696,0.404696,...,0.469317,0.469317,0.469317,0.469317,0.469317,0.469317,0.469317,0.000037,1.424117e-05,1.334943e-05
2020-01-09 05:00:00+00:00,2.524555e-07,0.000502,2020-01-09 05:00:00+00:00,0.402944,0.402944,0.402944,0.402944,0.402944,0.402944,0.402944,...,0.473052,0.473052,0.473052,0.473052,0.473052,0.473052,0.473052,0.000027,1.308552e-05,1.127573e-05


### AT THIS POINT OF THE NOTEBOOK I STOP ! MOVE TO ANALYSIS.PY(CONTAINS REGIME_SWITCHING_MFIV) OR MOVE TO TESTING.PY VARIANTS.

In [31]:

# CHECK NOT NA
[(outputs_by_year[2021]["q4"][[f"MFIV_{H}"]].notna().sum()) for H in HOURS]

[MFIV_2    2208
 dtype: int64,
 MFIV_3    2208
 dtype: int64,
 MFIV_4    2208
 dtype: int64,
 MFIV_5    2208
 dtype: int64,
 MFIV_6    2208
 dtype: int64,
 MFIV_7    2208
 dtype: int64,
 MFIV_8    2208
 dtype: int64,
 MFIV_9    2208
 dtype: int64,
 MFIV_10    2208
 dtype: int64,
 MFIV_11    2208
 dtype: int64,
 MFIV_12    2208
 dtype: int64,
 MFIV_13    2208
 dtype: int64,
 MFIV_14    2208
 dtype: int64,
 MFIV_15    2208
 dtype: int64,
 MFIV_16    2208
 dtype: int64,
 MFIV_17    2208
 dtype: int64,
 MFIV_18    2208
 dtype: int64,
 MFIV_19    2208
 dtype: int64,
 MFIV_20    2208
 dtype: int64,
 MFIV_21    2208
 dtype: int64,
 MFIV_22    2208
 dtype: int64,
 MFIV_23    2208
 dtype: int64,
 MFIV_24    2208
 dtype: int64,
 MFIV_25    2208
 dtype: int64,
 MFIV_26    2208
 dtype: int64,
 MFIV_27    2208
 dtype: int64,
 MFIV_28    2208
 dtype: int64,
 MFIV_29    2208
 dtype: int64,
 MFIV_30    2208
 dtype: int64,
 MFIV_31    2208
 dtype: int64,
 MFIV_32    2208
 dtype: int64,
 MFIV_33    2208

### GARCH regime switching

In [ ]:
sys.modules.pop("srcB", None)

<module 'srcA' (namespace) from ['/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/MFIV_calibration/srcA']>

In [23]:
# # Prepare the dictionaries. Designed to be overwritten

YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

idx_by_year = {y : {} for y in YEARS}
f_rollB_by_year = {y: {} for y in YEARS}
rolling_forecasts =  {y: {} for y in YEARS}
rolling_residuals = {y: {} for y in YEARS}
y_pred_by_year = {y: {} for y in YEARS}
y_true_by_year = {y: {} for y in YEARS}


In [ ]:
# EXPERIMENTAL 04
def regime_switching_mfiv(
    Xq_df,
    MFIV_COLS=["MFIV_191", "MFIV_382", "MFIV_573", "MFIV_764"],
    return_col="rt",
    target_col="rv",
    refit_every=1,
    min_improvement=0.02
):
    rv = Xq_df[target_col].astype(float)
    mfiv = Xq_df[list(MFIV_COLS)].astype(float)

    horizons = {
        col: int(col.split("_")[1])
        for col in MFIV_COLS
    }

    # For an MFIV forecast made at time t:
    #
    # matched_future_rv[col][t]
    #     = rv[t+1] + ... + rv[t+H]
    #
    # This is used only for historical forecast evaluation.
    matched_future_rv = {}

    for col, H in horizons.items():
        matched_future_rv[col] = (
            rv.shift(-1)
              .rolling(window=H, min_periods=H)
              .sum()
              .shift(-(H - 1))
        )

    f_rollB = []
    y_true = []
    idx = []
    regime_state = []
    selected_horizon = []

    selected_col = None

    # The longest horizon needs:
    # H hours for each completed outcome
    # plus H completed forecast-outcome observations.
    start = 2 * max(horizons.values())

    for i in range(start, len(Xq_df)):

        if selected_col is None or (i - start) % refit_every == 0:
            losses = {}

            for col, H in horizons.items():

                # At time i, rv[i] is not assumed available.
                # Therefore the latest completed H-hour outcome must
                # finish at or before i - 1.
                #
                # A forecast originating at u finishes at u + H.
                # Therefore u <= i - H - 1.
                history_end = i - H

                forecast_history = mfiv[col].iloc[:history_end]
                realised_history = matched_future_rv[col].iloc[:history_end]

                valid = (
                    forecast_history.notna()
                    & realised_history.notna()
                    & (forecast_history > 0)
                    & (realised_history > 0)
                )

                forecast_history = forecast_history.loc[valid].iloc[-H:]
                realised_history = realised_history.loc[valid].iloc[-H:]

                # Each horizon is evaluated using H completed
                # forecast-outcome pairs.
                if len(forecast_history) < H:
                    losses[col] = np.nan
                    continue

                ratio = realised_history / forecast_history

                # QLIKE divergence.
                qlike = ratio - np.log(ratio) - 1

                losses[col] = float(qlike.mean())

            losses = pd.Series(losses, dtype=float).dropna()

            if not losses.empty:
                best_col = losses.idxmin()
                best_loss = losses.loc[best_col]

                if selected_col is None:
                    selected_col = best_col

                elif selected_col not in losses.index:
                    selected_col = best_col

                else:
                    current_loss = losses.loc[selected_col]

                    # Avoid switching because of a negligible
                    # difference in historical loss.
                    if best_loss < current_loss * (1 - min_improvement):
                        selected_col = best_col

        if selected_col is None:
            continue

        forecast = mfiv[selected_col].iloc[i]

        if pd.isna(forecast) or forecast <= 0:
            continue

        matched_outcome = matched_future_rv[selected_col].iloc[i]

        f_rollB.append(float(forecast))

        # Future realised variance is used only as the evaluation target.
        # It is not used when selecting the regime at time i.
        y_true.append(
            float(matched_outcome)
            if pd.notna(matched_outcome)
            else np.nan
        )

        idx.append(i)
        regime_state.append(MFIV_COLS.index(selected_col))
        selected_horizon.append(selected_col)
   
    return (
        np.asarray(f_rollB, dtype=float),
        np.asarray(y_true, dtype=float),
        np.asarray(idx, dtype=int),
        np.asarray(regime_state, dtype=int),
        np.asarray(selected_horizon, dtype=object)
    )

This model is a dynamic MFIV horizon selector, not a GARCH model and not a regression model.

It:
- compares four MFIV forecasts: MFIV_191, MFIV_382, MFIV_573, and MFIV_764;
- treats each MFIV_H[t] as a forecast of cumulative realised variance over t+1 through t+H;
- evaluates each forecast only after that future realised variance is fully known;
- uses QLIKE loss: realised / forecast - log(realised / forecast) - 1;
- evaluates MFIV_H using its latest H valid, positive, completed forecast-outcome pairs;
- begins producing results only after 2 × max(H) rows;
- reassesses the selected horizon every row because refit_every=1;
- selects the MFIV horizon with the lowest historical average QLIKE loss;
- keeps the current horizon unless the new horizon improves loss by more than 2%;
- outputs the selected MFIV value as f_rollB;
- outputs its matched future cumulative realised variance as y_true, which may be NaN near the end;
- outputs the integer row positions as idx;
- outputs the selected horizon’s list index as regime_state;
- outputs the actual column name, such as MFIV_382, as selected_horizon.


This function’s job remains:
- select the best MFIV_H,
- output the selected MFIV forecast,
- output regime_state,
- output selected_horizon.

In [ ]:
YEARS = [2019, 2020, 2021]
QUARTERS = ["q1", "q2", "q3", "q4"]

for y in YEARS:
    for q in QUARTERS:
        Xq_df = Xq_by_year[y][q]

        f_rollB, y_true, idx, regime_state, selected_horizon = regime_switching_mfiv(Xq_df)

        idx_by_year[y][q] = idx
        f_rollB_by_year[y][q] = f_rollB
        rolling_forecasts[y][q] = f_rollB
        rolling_residuals[y][q] = y_true - f_rollB
        y_pred_by_year[y][q] = f_rollB
        y_true_by_year[y][q] = y_true

        outputs_by_year[y][q]["regime_switching_mfiv"] = pd.Series(f_rollB, index=Xq_df.index[idx])

        outputs_by_year[y][q]["regime_state"] = pd.Series(regime_state, index=Xq_df.index[idx] )

        outputs_by_year[y][q]["selected_horizon"] = pd.Series( selected_horizon, index=Xq_df.index[idx])


In [40]:
outputs_by_year[2021]["q4"]["selected_horizon"].nunique()

1

In [41]:
outputs_by_year[2020]["q4"].columns

Index(['rv', 'rt', 'date_time', 'garch', 'garch_mfiv', 'garch_mfiv_avg',
       'MFIV_191', 'MFIV_382', 'MFIV_573', 'MFIV_764', 'regime_switching_mfiv',
       'regime_state', 'selected_horizon'],
      dtype='str')

In [42]:
import pickle
with open("/Users/shah/CODE_BOOK_4/code_document/REPOSITORIES/data/IS/outputs_by_year/outputs_by_year_04.pkl", "wb") as f:
    pickle.dump(outputs_by_year, f)